In [ ]:
# --- Setup: imports and global matplotlib style ---
# Optional: # !pip install prophet statsmodels

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ponytail: fixed palette for hotel-themed plots (match Regression I notebook)
HOTEL_PALETTE = ['#1B3A5C', '#2E8B9A', '#E8A838', '#D45B5B', '#6B9E78', '#9B6BA8']
sns.set_theme(style='whitegrid', context='notebook', palette=HOTEL_PALETTE)
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12, 'axes.labelsize': 10})

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
%matplotlib inline


def mae_rmse(y_true, y_pred, name='model'):
    """Holdout report card for forecasts."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'model': name,
        'MAE': round(float(mean_absolute_error(y_true, y_pred)), 2),
        'RMSE': round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 2),
    }

prophet_row = None
ridge_row = None


<div class="jumbotron">
    <h1 class="display-1">Regression II: Time Series</h1>
    <hr class="my-4">
    <p class="lead">Instructor: Dr. Yan Li</p>
</div>


## Agenda

1. **The business problem** — staffing and rates from daily arrivals
2. **Why time series is different** — order, lag, leakage
3. **Classical decomposition** — peel the series into trend, season, residual
4. **Prophet** — calendar-aware forecast with uncertainty
5. **ML lag models** — turn yesterday into features
6. **Model comparison** — which method to ship


## 1. The business problem

> **Opening puzzle:** *Friday arrivals spike every week, but next month has a city marathon.
> How much of demand is “always Friday,” and how much is a one-off event — so we can staff
> the front desk 14 days ahead?*

A hotel group needs a forecast of <span class="mark">daily arrivals</span> — think of it as
the hotel’s **daily guest count**, the number of rooms that fill each calendar day.

Revenue managers need a model to:

- **Explain** which part of demand is slow trend vs weekly season vs noise
- **Forecast** arrivals for the next 14–30 days with a credible band
- **Flag** unusual days (events, shocks) that break the usual pattern


### The data: hotel booking demand → daily arrivals

We start from the public **Hotel Booking Demand** dataset (Antonio, de Almeida &amp; Nunes, 2019).
Each row is one booking. For forecasting we **aggregate** bookings to a daily arrival count.

<div class="alert alert-info">
    <strong>Dataset:</strong> <code>hotel_bookings.csv</code> &middot; ~119k bookings &middot; 32 columns.<br>
    Series target <code>arrivals</code> — a <strong>daily count</strong>, so this is a
    <em>forecasting</em> problem (ordered in time, not a shuffled regression table).
</div>


### Series dictionary

| Field | Meaning | Role |
|---|---|---|
| `arrival_date` | Calendar day of guest arrival | Time index (`ds`) |
| `arrivals` | Number of bookings arriving that day | **Target** (`y`) |
| `mean_adr` | Average daily rate that day (EUR) | Optional context |
| `cancel_rate` | Share of cancelled bookings that day | Optional context |

<div class="alert alert-warning">
    <strong>Rule:</strong> never randomly shuffle days for train/test. Time has a direction.
</div>


### Load and build the daily series

We parse arrival year / month / day, drop broken dates, and count bookings per day.
A short holdout at the **end** of the series (last 30 days) is our test window.


In [ ]:
# --- Load hotel bookings and aggregate to daily arrivals ---
df = pd.read_csv('./data/analysis/hotel_bookings.csv')

month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12,
}
d = df.copy()
d['arrival_month_num'] = d['arrival_date_month'].map(month_map)
d['arrival_date'] = pd.to_datetime(
    dict(year=d['arrival_date_year'], month=d['arrival_month_num'],
         day=d['arrival_date_day_of_month']),
    errors='coerce',
)

daily = (
    d.dropna(subset=['arrival_date'])
     .groupby('arrival_date')
     .agg(arrivals=('hotel', 'size'),
          cancel_rate=('is_canceled', 'mean'),
          mean_adr=('adr', 'mean'))
     .sort_index()
)

# Time-respecting holdout: last 30 days
H = 30
train, test = daily.iloc[:-H].copy(), daily.iloc[-H:].copy()

print(f'Days total: {len(daily):,} | Train: {len(train):,} | Test (holdout): {len(test):,}')
print(f'Range: {daily.index.min().date()} → {daily.index.max().date()}')
daily[['arrivals']].describe().round(1)


### Explore arrivals before modeling

Before any model, look at the **shape** of demand. Weekly spikes and slow drifts tell us
whether decomposition and calendar models are plausible.


In [ ]:
# --- EDA: daily arrivals and day-of-week pattern ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(daily.index, daily['arrivals'], color=HOTEL_PALETTE[1], lw=0.8)
axes[0].axvline(test.index.min(), color=HOTEL_PALETTE[3], ls='--', lw=2, label='holdout start')
axes[0].set_title('Daily arrivals over time')
axes[0].set_ylabel('Arrivals'); axes[0].legend()

dow = daily.copy()
dow['dow'] = dow.index.day_name()
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.boxplot(data=dow, x='dow', y='arrivals', order=order, ax=axes[1], color=HOTEL_PALETTE[2])
axes[1].set_title('Arrivals by day of week'); axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()


## 2. Why time series is different

In Regression I, rows were bookings we could shuffle. Here each day **follows** the day before.
That changes how we split data, how we build features, and what “error” means.


<dl class="row alert alert-success">
    <dt class="col-md-3">Temporal order</dt>
    <dd class="col-md-auto">
        <strong>Definition:</strong> Observations sit in a fixed sequence; swapping them destroys meaning.
        <br><br>
        <strong>Explanation:</strong> Tuesday follows Monday. If you shuffle days like a deck of cards,
        the hotel calendar story disappears — weekends no longer follow weekdays.
    </dd>
</dl>


<dl class="row alert alert-info">
    <dt class="col-md-3">Lag</dt>
    <dd class="col-md-auto">
        <strong>Definition:</strong> A past value used as a predictor of the present.
        <br><br>
        <strong>Explanation:</strong> Yesterday’s occupancy is a clue for today’s walk-ins — like
        remembering last Friday when you plan this Friday’s staffing.
    </dd>
</dl>


<dl class="row alert alert-warning">
    <dt class="col-md-3">Leakage</dt>
    <dd class="col-md-auto">
        <strong>Definition:</strong> Training with information that would not be known at forecast time.
        <br><br>
        <strong>Explanation:</strong> Using tonight’s midnight cancellation total to “predict” this morning’s
        arrivals is cheating — the front desk never had that number when the day started.
    </dd>
</dl>


<div class="alert alert-warning">
    <strong>Why this matters:</strong> a randomly shuffled train/test split can look brilliant in class
    and fail in the hotel. Always cut time: train on the past, test on the future.
</div>


## 3. Classical decomposition

**Goal:** peel the arrival series into pieces managers can name — slow level, weekly rhythm, leftovers.


### Principles of classical decomposition

Think of daily arrivals as a song mixed from three tracks. Decomposition **unmixes** the song
into bass (trend), drums (season), and leftover noise.

**Additive form** (seasonal swings look roughly constant in size):

$$y_t = T_t + S_t + R_t$$

**Multiplicative form** (seasonal swings grow when the level grows):

$$y_t = T_t \times S_t \times R_t$$

In plain English:

- $T_t$ — slow level (trend / cycle): “are we busier this year?”
- $S_t$ — repeating calendar pattern: “Fridays run hotter”
- $R_t$ — leftover: noise and one-off shocks


*Decomposition is like lifting three transparent sheets off the same hotel calendar:
one for the slow climb, one for the weekly stamp, one for the mess that remains.*

**Why hotels care:** managers need language for “is demand rising?” versus “is it just Friday?”
Mixing those stories leads to bad staffing and bad rates.


### Decomposition in code: form and knobs

**Function:** `statsmodels.tsa.seasonal.seasonal_decompose`

| Parameter | Role (plain English) | Undergrad default |
|---|---|---|
| `model` | Add pieces vs multiply pieces | `'additive'` when weekly swings look similar in height |
| `period` | How many days make one season | `7` for daily data with a weekly pattern |
| `extrapolate_trend` | Fill trend at the edges | `'freq'` if edge NaNs annoy the plot |


### Decomposition demo: peel the training series

We decompose the **training** window only (no peeking into the holdout).


In [ ]:
# --- Classical additive decomposition (weekly period) ---
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
except ImportError:
    seasonal_decompose = None
    print('statsmodels not installed. Install statsmodels to run decomposition.')

if seasonal_decompose is not None:
    decomp = seasonal_decompose(train['arrivals'], model='additive', period=7, extrapolate_trend='freq')
    fig = decomp.plot()
    fig.set_size_inches(10, 8)
    fig.suptitle('Additive decomposition of daily arrivals (train)', y=1.02)
    plt.tight_layout(); plt.show()


### Reading the decomposition

- **Trend panel:** smooth climb or dip — the slow story for capacity planning
- **Seasonal panel:** repeating weekly wave — Friday/weekend stamp
- **Residual panel:** leftover spikes — candidates for events, data issues, or true shocks

If residuals still show a clear weekly shape, `period` or `model` may be wrong.


### Decomposition section summary

| Aspect | Summary |
|---|---|
| **What** | Split $y_t$ into trend, season, residual |
| **When** | First look: explain the calendar shape of demand |
| **Watch out** | Weak forward forecast path; holidays hide in residuals |
| **Next** | Prophet keeps the same story and ships a forecast with bands |

<div class="alert alert-warning">
    <strong>Why this matters:</strong> classical decomposition shows <em>what the pieces look like</em>,
    but it does not easily add named holidays or project pieces forward with uncertainty bands.
    <strong>Prophet</strong> keeps “trend + season + leftover” and turns it into a forecast managers can ship.
</div>


## 4. Prophet regression (calendar-aware forecast)

**Goal:** forecast arrivals with trend, seasonality, optional holidays, and an uncertainty band.


### Why decomposition alone is not enough

Decomposition is a microscope. Revenue managers also need a **telescope**: numbers for next week,
plus a band that says how unsure we are.

Prophet is a calendar-aware sketch artist: it draws a flexible trend, stamps repeating seasonal
shapes, then adds holiday bumps you name on the calendar.


### Principles of Prophet

Prophet models the series as an **additive** (or multiplicative) sum:

$$\hat{y}(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

| Symbol | Plain English |
|---|---|
| $g(t)$ | Trend — piecewise linear (or logistic) level |
| $s(t)$ | Seasonality — weekly / yearly repeating shapes |
| $h(t)$ | Holidays / known events |
| $\varepsilon_t$ | Noise |

*Prophet annotates a hotel calendar: flexible ruler for trend, rubber stamp for week/year,
sticky notes for marathon weekend and New Year.*


**Why hotels care:** city marathons and holidays break a pure “every Friday” story.
Prophet lets managers **name the dates** instead of hoping residuals absorb them.


### Prophet in code: form and knobs

**API:** `Prophet` → `fit` → `make_future_dataframe` → `predict`  
**Data contract:** columns `ds` (datetime) and `y` (numeric)

| Parameter | Role (plain English) | Undergrad default |
|---|---|---|
| `growth` | How the long-run level moves | `'linear'` |
| `yearly_seasonality` | Repeat-every-year pattern | `True` or `'auto'` |
| `weekly_seasonality` | Weekend vs weekday | `True` |
| `daily_seasonality` | Intra-day (rare for daily totals) | `False` |
| `changepoint_prior_scale` | How freely the trend can bend | start at `0.05` |
| `seasonality_prior_scale` | How strong seasonal waves can be | default `10.0` |
| `interval_width` | Width of uncertainty band | `0.8` |


### Prophet demo: fit on train, score on holdout

If `prophet` is not installed, the cell prints a clear skip message (uncomment the pip line in Setup).


In [ ]:
# --- Prophet forecast on daily arrivals ---
prophet_row = None
try:
    from prophet import Prophet
except ImportError:
    Prophet = None
    print('Prophet not installed. Optional: uncomment pip install in the Setup cell.')

if Prophet is not None:
    pdf = train.reset_index().rename(columns={'arrival_date': 'ds', 'arrivals': 'y'})[['ds', 'y']]

    m = Prophet(
        growth='linear',
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        interval_width=0.8,
    )
    m.fit(pdf)

    future = m.make_future_dataframe(periods=H, freq='D')
    fc = m.predict(future)

    # Align holdout predictions
    fc_h = fc.set_index('ds').loc[test.index, ['yhat', 'yhat_lower', 'yhat_upper']]
    prophet_row = mae_rmse(test['arrivals'], fc_h['yhat'], name='Prophet')
    print(prophet_row)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(train.index, train['arrivals'], color=HOTEL_PALETTE[0], lw=0.7, alpha=0.7, label='train')
    ax.plot(test.index, test['arrivals'], color=HOTEL_PALETTE[1], lw=2, label='actual holdout')
    ax.plot(fc_h.index, fc_h['yhat'], color=HOTEL_PALETTE[3], lw=2, label='Prophet yhat')
    ax.fill_between(fc_h.index, fc_h['yhat_lower'], fc_h['yhat_upper'],
                    color=HOTEL_PALETTE[3], alpha=0.2, label='80% band')
    ax.set_title('Prophet: holdout forecast vs actual arrivals')
    ax.set_ylabel('Arrivals'); ax.legend()
    plt.tight_layout(); plt.show()


### Prophet components (trend and weekly stamp)

Component plots show the same story decomposition told — now inside a forecasting model.


In [ ]:
# --- Prophet component plots (trend + seasonality) ---
if Prophet is not None:
    fig2 = m.plot_components(fc)
    fig2.set_size_inches(10, 8)
    plt.tight_layout(); plt.show()


### Dial demo: changepoint prior (trend flexibility)

Small `changepoint_prior_scale` → stiff trend. Large → wiggly trend that chases noise.
Try three settings and watch holdout MAE.


In [ ]:
# --- Prophet: changepoint_prior_scale sensitivity on holdout MAE ---
if Prophet is not None:
    rows = []
    for cps in [0.001, 0.05, 0.5]:
        m_i = Prophet(
            growth='linear', yearly_seasonality=True, weekly_seasonality=True,
            daily_seasonality=False, changepoint_prior_scale=cps, interval_width=0.8,
        )
        m_i.fit(pdf)
        fc_i = m_i.predict(future)
        yhat_i = fc_i.set_index('ds').loc[test.index, 'yhat']
        rows.append(mae_rmse(test['arrivals'], yhat_i, name=f'Prophet(cps={cps})'))
    pd.DataFrame(rows)


### Reading the Prophet output

- **yhat:** point forecast staffing planners can put on a sheet
- **Band (yhat_lower / yhat_upper):** wider band = more uncertainty — plan buffer staff
- **Components:** confirm weekly shape matches the boxplot from EDA
- **cps dial:** too large → nervous trend; too small → stubborn trend

### Prophet section summary

| Aspect | Summary |
|---|---|
| **What** | Calendar-aware additive forecast with uncertainty |
| **When** | Strong seasonality + known events; fast business forecast |
| **Watch out** | Needs history; not a causal pricing engine |
| **Next** | ML lag models when many advance-known drivers matter |

<div class="alert alert-warning">
    <strong>Why this matters:</strong> Prophet is strong on calendar structure. When you also have
    many extra drivers known in advance (promos, segment mix), an <strong>ML lag model</strong>
    turns yesterday’s numbers into a supervised table.
</div>


## 5. ML lag models (supervised forecasting)

**Goal:** reframe “predict tomorrow” as ordinary regression on past values — then reuse Ridge.


### Principles of lag features

Turn the series into a table a Regression I model already understands:

$$y_t = f(y_{t-1}, y_{t-7}, \ldots, x_t) + \varepsilon_t$$

| Piece | Plain English |
|---|---|
| $y_{t-1}$ | Yesterday’s arrivals |
| $y_{t-7}$ | Same weekday last week |
| $x_t$ | Advance-known drivers only (e.g. `is_weekend`) |

**Analogy:** packing a suitcase for tomorrow using what you packed last week — not using
clothes you will only buy after the trip (that would be leakage).


### Ridge-on-lags in code: form and knobs

**Function:** `sklearn.linear_model.Ridge`

| Parameter | Role (plain English) | Undergrad default |
|---|---|---|
| `alpha` | L2 complexity tax (from Regression I) | try `1.0` |
| lag set | Which past days enter $f$ | `1` and `7` are a minimal hotel set |
| time split | Train on past rows only | last $H$ days held out |

**Why hotels care:** when the story is “recent demand + weekday,” lag Ridge is simple to
maintain and ties back to regularization you already know.


### ML demo: build lags, fit Ridge, score holdout


In [ ]:
# --- Lag features + Ridge (time-ordered holdout) ---
feat = daily[['arrivals']].copy()
feat['lag_1'] = feat['arrivals'].shift(1)
feat['lag_7'] = feat['arrivals'].shift(7)
feat['is_weekend'] = (feat.index.dayofweek >= 5).astype(int)
feat = feat.dropna()

# Same calendar holdout window as Prophet (last H days that exist in feat)
feat_train = feat.loc[feat.index < test.index.min()]
feat_test = feat.loc[feat.index >= test.index.min()]

X_cols = ['lag_1', 'lag_7', 'is_weekend']
ridge = Ridge(alpha=1.0)
ridge.fit(feat_train[X_cols], feat_train['arrivals'])
ridge_pred = ridge.predict(feat_test[X_cols])

ridge_row = mae_rmse(feat_test['arrivals'], ridge_pred, name='Ridge(lags)')
print(ridge_row)
print('Coefficients:', dict(zip(X_cols, np.round(ridge.coef_, 3))))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(feat_test.index, feat_test['arrivals'], color=HOTEL_PALETTE[1], lw=2, label='actual')
ax.plot(feat_test.index, ridge_pred, color=HOTEL_PALETTE[4], lw=2, label='Ridge lag forecast')
ax.set_title('Ridge on lags: holdout vs actual')
ax.set_ylabel('Arrivals'); ax.legend()
plt.tight_layout(); plt.show()


### Reading the lag-Ridge output

- **lag_7 coefficient:** strength of the weekly echo (Friday remembers last Friday)
- **lag_1 coefficient:** short-term persistence
- **Holdout plot:** systematic miss on event weeks → need holiday features or Prophet events

### ML lag section summary

| Aspect | Summary |
|---|---|
| **What** | Supervised regression on past $y$ and advance-known $x$ |
| **When** | Many drivers; team already ships sklearn pipelines |
| **Watch out** | Leakage; weaker pure calendar story than Prophet |
| **Next** | Compare methods on the **same** holdout window |


## 6. Model comparison: which one to ship

All methods share the same hotel arrival series and the same last-30-day holdout.
Naive seasonal baseline (repeat last week) keeps everyone honest.


In [ ]:
# --- Holdout comparison: seasonal naive + available models ---
# Seasonal naive: predict y_t with y_{t-7}
naive_pred = []
hist = train['arrivals'].copy()
for dt in test.index:
    naive_pred.append(float(hist.iloc[-7]))
    hist.loc[dt] = test.loc[dt, 'arrivals']
naive_row = mae_rmse(test['arrivals'], naive_pred, name='SeasonalNaive(lag7)')

rows = [naive_row]
if prophet_row is not None:
    rows.append(prophet_row)
if ridge_row is not None:
    rows.append(ridge_row)

cmp = pd.DataFrame(rows).sort_values('MAE')
cmp


### Contrast card

| Aspect | Decomposition | Prophet | ML lags |
|---|---|---|---|
| **Goal** | Explain pieces | Forecast + events + band | Forecast with feature soup |
| **Strength** | Transparent $T,S,R$ | Holidays, uncertainty | Flexible $f(\cdot)$, sklearn-ready |
| **Weakness** | Weak forward path | Less “extra regressor” flexibility | Leakage risk; thinner calendar story |

**Manager recommendation:**

1. **Accuracy:** pick the lowest holdout MAE/RMSE on *future* days, not training fit
2. **Interpretability:** decomposition + Prophet components for the exec deck
3. **Maintenance:** lag Ridge if the team already owns sklearn; Prophet if the calendar dominates

<div class="alert alert-success">
    <strong>Rule of thumb:</strong> start with decomposition to <em>see</em> the series,
    use Prophet when the calendar matters, add ML lags when you have extra advance-known
    drivers — never shuffle time.
</div>


### Notebook takeaway

| Aspect | Summary |
|---|---|
| **What** | Forecasting daily hotel arrivals with decomposition, Prophet, and lag ML |
| **When** | Staffing / rate decisions need tomorrow’s demand, not shuffled booking rows |
| **Watch out** | Leakage, random splits, trusting training error |
| **Link to Regression I** | Ridge returns — now the “features” are lags and calendar flags |
